### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [6]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "baseline" / "XGBoost_thres0.9_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [7]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [8]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [9]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [10]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.29,,,,,
1,0,cf_1,,,,,,,17.1658,,,0.1118,1,True,0.026,0.004
2,0,cf_2,,,,7,,,18.9745,,,0.1257,2,False,0.026,0.0707
3,0,cf_3,3,,,,,,18.0887,,,0.1801,2,True,0.026,0.0064
4,0,cf_4,,,,,,,18.0858,3,,0.1089,2,True,0.026,0.0005
5,0,cf_5,,,,,,,18.0569,5,,0.1464,2,True,0.026,0.001
6,0,cf_6,,,,7,,,17.8165,,,0.1969,2,False,0.026,0.0424
7,0,cf_7,,,,6,,,17.5744,,,0.1701,2,True,0.026,0.0055
8,0,cf_8,,,6,,,,16.9514,,,0.25,2,True,0.026,0.005
9,0,cf_9,,,6,7,,,18.9745,,,0.2507,3,False,0.026,0.1428


### Filtering Valid CFs

In [11]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.29,,,,,
9,0,cf_1,,,,,,,17.1658,,,0.1118,1,True,0.026,0.004
10,0,cf_3,3,,,,,,18.0887,,,0.1801,2,True,0.026,0.0064
11,0,cf_4,,,,,,,18.0858,3,,0.1089,2,True,0.026,0.0005
12,0,cf_5,,,,,,,18.0569,5,,0.1464,2,True,0.026,0.001
13,0,cf_7,,,,6,,,17.5744,,,0.1701,2,True,0.026,0.0055
14,0,cf_8,,,6,,,,16.9514,,,0.25,2,True,0.026,0.005
15,0,cf_10,,,,7,,,18.9745,7,,0.2507,3,True,0.026,0.0038
1,1,xin,3,4,1,2,3,0,22.3757,0,1.1,,,,,
16,1,cf_2,2,,,,1,,22.3757,,,0.2546,3,True,0.2497,0.0361


### select one CF

In [12]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.29,,,,,
9,0,cf_1,,,,,,,17.1658,,,0.1118,1,True,0.026,0.004
1,1,xin,3,4,1,2,3,0,22.3757,0,1.1,,,,,
10,1,cf_3,,3,,,,,22.3757,6,,0.2368,3,True,0.2497,0.1213
2,2,xin,5,3,1,1,4,0,22.694,7,1.16,,,,,
11,2,cf_4,,,,3,2,,22.6757,,,0.25,3,True,0.218,0.0426
3,3,xin,3,3,6,6,2,0,24.3418,1,1.16,,,,,
12,3,cf_4,,1,,,,,24.3375,,,0.1286,2,True,0.0653,0.0042
4,4,xin,5,4,2,7,1,0,21.2585,3,1.25,,,,,
13,4,cf_2,,,6,,,,21.2585,,,0.2368,2,True,0.0811,0.0327


In [13]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.004
Valid: True
Changes:
  bmi: 18.9866 → 17.1658

--- cf_3 ---
Predicted risk: 0.0064
Valid: True
Changes:
  etfruit: 4 → 3
  bmi: 18.9866 → 18.0887

--- cf_4 ---
Predicted risk: 0.0005
Valid: True
Changes:
  bmi: 18.9866 → 18.0858
  dosprt: 0 → 3

--- cf_5 ---
Predicted risk: 0.001
Valid: True
Changes:
  bmi: 18.9866 → 18.0569
  dosprt: 0 → 5

--- cf_7 ---
Predicted risk: 0.0055
Valid: True
Changes:
  alcfreq: 4 → 6
  bmi: 18.9866 → 17.5744

--- cf_8 ---
Predicted risk: 0.005
Valid: True
Changes:
  cgtsmok: 3 → 6
  bmi: 18.9866 → 16.9514

--- cf_10 ---
Predicted risk: 0.0038
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original i